# OpsSim-AI: Before vs After LoRA Comparison

This notebook compares a **base model** (before training) against the **base + trained LoRA adapter** (after GRPO training) on cascading DevOps incident scenarios.

It reuses the exact prompt-building, state-replay, and parsing logic from the OpsSim-AI repo.

## 1. Configuration

Set your base model, adapter repo, and HF token below.

In [ ]:
# ====== CONFIGURE THESE ======
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "meancodi/opssim-qwen-3b-v1"
HF_TOKEN = ""  # paste your token or set via env

# Inference settings
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0  # greedy for reproducibility
TOP_P = 1.0
PRECISION = "bf16"  # bf16 | fp16 | fp32

# How many scenario+step pairs to test
NUM_EXAMPLES = 5

## 2. Install Dependencies & Clone Repo

In [ ]:
!pip install -q torch transformers accelerate peft huggingface_hub
!pip install -q jmespath pydantic

import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Clone the repo to get tasks/cascade.json and helper modules
REPO_DIR = "OpsSim-AI"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/nithishgouds/Meta-x-OpenEnv-x-Pytorch-Hackathon.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 3. Import Repo Utilities

We reuse the exact functions from the training pipeline — no rewriting.

In [ ]:
import copy
import json
from typing import Any
from collections import OrderedDict

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Reuse directly from the repo ---
from generate_sft_dataset import (
    load_scenarios,
    apply_effects,
    build_prompt,
    build_assistant,
    evaluate_condition,
    detect_anomalies,
    find_agent_for_action,
    classify_unsafe_actions,
)

from run_trained_inference import (
    extract_json,
    replay_state,
    render_chat,
)

print("All repo utilities imported successfully.")

## 4. Load Scenarios

In [ ]:
scenarios = load_scenarios("tasks/cascade.json")
print(f"Loaded {len(scenarios)} scenarios:")
for s in scenarios:
    optimal = s.get("optimal_solution_path", [])
    print(f"  {s['scenario_id']}  ({len(optimal)} steps)")

## 5. Build Test Examples

Pick diverse scenario+step combinations across different scenarios and step indices.

In [ ]:
import random
random.seed(42)

test_examples = []
# Pick examples from different scenarios at different steps
candidates = []
for scenario in scenarios:
    optimal = scenario.get("optimal_solution_path", []) or []
    for step_idx in range(1, len(optimal) + 1):
        candidates.append((scenario, step_idx))

selected = random.sample(candidates, min(NUM_EXAMPLES, len(candidates)))

for scenario, step_idx in selected:
    optimal = scenario.get("optimal_solution_path", []) or []
    total_steps = len(optimal)
    completed = optimal[:step_idx - 1]
    state, rewards = replay_state(scenario, completed)
    gold_action = optimal[step_idx - 1]
    gold_agent, gold_domain = find_agent_for_action(
        gold_action, scenario.get("action_domains", {})
    )

    prompt = build_prompt(
        scenario=scenario,
        state=state,
        step_idx=step_idx,
        total_steps=total_steps,
        completed=completed,
        completed_rewards=rewards,
        candidate=None,
    )

    gold_response = build_assistant(
        scenario=scenario,
        state=state,
        step_idx=step_idx,
        total_steps=total_steps,
        completed=completed,
        completed_rewards=rewards,
        action=gold_action,
    )

    unsafe_actions = [
        u["action"] for u in classify_unsafe_actions(scenario, state, set(completed), gold_action)
    ]

    test_examples.append({
        "scenario_id": scenario["scenario_id"],
        "step_idx": step_idx,
        "total_steps": total_steps,
        "prompt": prompt,
        "gold_action": gold_action,
        "gold_agent": gold_agent,
        "gold_response": gold_response,
        "unsafe_actions": unsafe_actions,
    })

print(f"Built {len(test_examples)} test examples:")
for ex in test_examples:
    print(f"  {ex['scenario_id']}  step {ex['step_idx']}/{ex['total_steps']}  gold={ex['gold_action']}")

## 6. Load Models

Load the base model ("before") and the base + adapter ("after").

In [ ]:
torch_dtype = {
    "bf16": torch.bfloat16,
    "fp16": torch.float16,
    "fp32": torch.float32,
}[PRECISION]

print(f"Loading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model ({PRECISION})...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

print(f"Loading adapter from {ADAPTER_REPO}...")
adapter_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
adapter_model.eval()

print("Both models loaded.")

## 7. Inference Helper

In [ ]:
def generate(model, prompt: str) -> str:
    """Generate a response from the model given a prompt."""
    rendered = render_chat(tokenizer, prompt)
    inputs = tokenizer(rendered, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=TEMPERATURE > 0.0,
            temperature=TEMPERATURE if TEMPERATURE > 0.0 else None,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("Inference helper ready.")

## 8. Metrics

In [ ]:
REQUIRED_KEYS = {"analysis", "plan", "next_action", "target_agent", "reasoning", "confidence"}

def evaluate_response(raw_output: str, example: dict) -> dict:
    """Score a single model response against the gold answer."""
    parsed = extract_json(raw_output)

    metrics = {
        "valid_json": parsed is not None,
        "has_all_keys": False,
        "correct_action": False,
        "correct_agent": False,
        "is_unsafe": False,
        "predicted_action": None,
        "predicted_agent": None,
    }

    if parsed is None:
        return metrics

    metrics["has_all_keys"] = REQUIRED_KEYS.issubset(parsed.keys())
    predicted_action = parsed.get("next_action", "")
    predicted_agent = parsed.get("target_agent", "")
    metrics["predicted_action"] = predicted_action
    metrics["predicted_agent"] = predicted_agent
    metrics["correct_action"] = predicted_action == example["gold_action"]
    metrics["correct_agent"] = predicted_agent == example["gold_agent"]
    metrics["is_unsafe"] = predicted_action in example["unsafe_actions"]

    return metrics

print("Metrics helper ready.")

## 9. Run Before vs After Comparison

In [ ]:
results = []

for i, example in enumerate(test_examples):
    print(f"\n{'='*80}")
    print(f"Example {i+1}/{len(test_examples)}")
    print(f"Scenario: {example['scenario_id']}")
    print(f"Step: {example['step_idx']}/{example['total_steps']}")
    print(f"Gold action: {example['gold_action']}  (agent: {example['gold_agent']})")
    print(f"{'='*80}")

    # --- Before (base model, adapter disabled) ---
    print("\n[BEFORE] Generating with base model...")
    adapter_model.disable_adapter_layers()
    before_raw = generate(adapter_model, example["prompt"])
    before_metrics = evaluate_response(before_raw, example)

    # --- After (adapter enabled) ---
    print("[AFTER]  Generating with base + adapter...")
    adapter_model.enable_adapter_layers()
    after_raw = generate(adapter_model, example["prompt"])
    after_metrics = evaluate_response(after_raw, example)

    results.append({
        "example": example,
        "before_raw": before_raw,
        "after_raw": after_raw,
        "before_metrics": before_metrics,
        "after_metrics": after_metrics,
    })

    # --- Side-by-side output ---
    print(f"\n--- BEFORE (base only) ---")
    print(before_raw[:500])
    print(f"\n--- AFTER (base + adapter) ---")
    print(after_raw[:500])

    print(f"\n--- Metrics ---")
    print(f"{'Metric':<20} {'Before':<15} {'After':<15}")
    print(f"{'-'*50}")
    for key in ["valid_json", "has_all_keys", "correct_action", "correct_agent", "is_unsafe"]:
        bval = str(before_metrics[key])
        aval = str(after_metrics[key])
        marker = ""
        if key == "is_unsafe":
            marker = " !!" if after_metrics[key] else ""
        else:
            marker = " ✓" if after_metrics[key] and not before_metrics[key] else ""
        print(f"{key:<20} {bval:<15} {aval:<15}{marker}")
    print(f"{'predicted_action':<20} {str(before_metrics['predicted_action']):<15} {str(after_metrics['predicted_action']):<15}")
    print(f"{'predicted_agent':<20} {str(before_metrics['predicted_agent']):<15} {str(after_metrics['predicted_agent']):<15}")

print(f"\n\nDone! Ran {len(results)} comparisons.")

## 10. Aggregate Summary

In [ ]:
n = len(results)
if n == 0:
    print("No results to summarize.")
else:
    summary = {"before": {}, "after": {}}
    for phase in ["before", "after"]:
        key = f"{phase}_metrics"
        summary[phase]["valid_json_rate"] = sum(1 for r in results if r[key]["valid_json"]) / n
        summary[phase]["has_all_keys_rate"] = sum(1 for r in results if r[key]["has_all_keys"]) / n
        summary[phase]["action_accuracy"] = sum(1 for r in results if r[key]["correct_action"]) / n
        summary[phase]["agent_accuracy"] = sum(1 for r in results if r[key]["correct_agent"]) / n
        summary[phase]["unsafe_rate"] = sum(1 for r in results if r[key]["is_unsafe"]) / n

    print(f"\n{'='*60}")
    print(f"  AGGREGATE SUMMARY  ({n} examples)")
    print(f"{'='*60}")
    print(f"{'Metric':<25} {'Before':<15} {'After':<15} {'Delta':<10}")
    print(f"{'-'*65}")
    for metric in ["valid_json_rate", "has_all_keys_rate", "action_accuracy", "agent_accuracy", "unsafe_rate"]:
        b = summary["before"][metric]
        a = summary["after"][metric]
        delta = a - b
        direction = "↑" if delta > 0 else ("↓" if delta < 0 else "→")
        # For unsafe_rate, lower is better
        if metric == "unsafe_rate":
            direction = "↓ ✓" if delta < 0 else ("↑ !!" if delta > 0 else "→")
        print(f"{metric:<25} {b:>6.1%}{'':>8} {a:>6.1%}{'':>8} {delta:>+6.1%} {direction}")

    print(f"\nInterpretation:")
    print(f"  ↑ = improved after training (higher is better)")
    print(f"  ↓ ✓ on unsafe_rate = improved (lower is better)")

## 11. Detailed Side-by-Side (Prettified)

Show each example's gold vs before vs after as formatted JSON.

In [ ]:
for i, r in enumerate(results):
    ex = r["example"]
    print(f"\n{'#'*80}")
    print(f"# Example {i+1}: {ex['scenario_id']}  step {ex['step_idx']}/{ex['total_steps']}")
    print(f"# Gold: {ex['gold_action']} → {ex['gold_agent']}")
    print(f"{'#'*80}")

    # Gold
    gold_parsed = json.loads(ex["gold_response"])
    print(f"\n--- GOLD (expected) ---")
    print(json.dumps(gold_parsed, indent=2))

    # Before
    before_parsed = extract_json(r["before_raw"])
    print(f"\n--- BEFORE (base model) ---")
    if before_parsed:
        print(json.dumps(before_parsed, indent=2))
    else:
        print(f"[INVALID JSON] Raw output:\n{r['before_raw'][:300]}")

    # After
    after_parsed = extract_json(r["after_raw"])
    print(f"\n--- AFTER (base + adapter) ---")
    if after_parsed:
        print(json.dumps(after_parsed, indent=2))
    else:
        print(f"[INVALID JSON] Raw output:\n{r['after_raw'][:300]}")

    # Quick verdict
    bm, am = r["before_metrics"], r["after_metrics"]
    verdict = []
    if am["correct_action"] and not bm["correct_action"]:
        verdict.append("Action FIXED by adapter")
    elif am["correct_action"] and bm["correct_action"]:
        verdict.append("Action correct in both")
    elif not am["correct_action"] and bm["correct_action"]:
        verdict.append("Action REGRESSED")
    else:
        verdict.append("Action wrong in both")

    if am["valid_json"] and not bm["valid_json"]:
        verdict.append("JSON FIXED")
    if bm["is_unsafe"] and not am["is_unsafe"]:
        verdict.append("Unsafe action AVOIDED")
    if am["is_unsafe"]:
        verdict.append("⚠ UNSAFE action chosen")

    print(f"\nVerdict: {' | '.join(verdict)}")

## 12. Cleanup

Free GPU memory if needed.

In [ ]:
del adapter_model, base_model
torch.cuda.empty_cache()
print("GPU memory freed.")